Алгоритм Леска

In [ ]:
import nltk
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet as wn

nltk.download('punkt')
nltk.download('wordnet')

In [23]:
text1 = "At night, a bat flew through the sky, flapping its wings and hunting insects."
text2 = "Tom forgot his bat on the cricket field."

words1 = word_tokenize(text1)
words2 = word_tokenize(text2)

In [1]:
from nltk.corpus import wordnet as wn #посмотрим на пример из тезауруса

for syn in wn.synsets('bat', pos=wn.NOUN):
    print(f"Synset name: {syn.name()}")
    print(f"  Lemmas (синонимы): {syn.lemma_names()}")
    print(f"  Definition: {syn.definition()}")
    print(f"  Examples: {syn.examples()}")
    print()


Synset name: bat.n.01
  Lemmas (синонимы): ['bat', 'chiropteran']
  Definition: nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate
  Examples: []

Synset name: bat.n.02
  Lemmas (синонимы): ['bat', 'at-bat']
  Definition: (baseball) a turn trying to get a hit
  Examples: ['he was at bat when it happened', 'he got four hits in four at-bats']

Synset name: squash_racket.n.01
  Lemmas (синонимы): ['squash_racket', 'squash_racquet', 'bat']
  Definition: a small racket with a long handle used for playing squash
  Examples: []

Synset name: cricket_bat.n.01
  Lemmas (синонимы): ['cricket_bat', 'bat']
  Definition: the club used in playing cricket
  Examples: ['a cricket bat has a narrow handle and a broad flat end for hitting']

Synset name: bat.n.05
  Lemmas (синонимы): ['bat']
  Definition: a club used for hitting a ball in various games
  Examples: []



In [24]:
sense1 = lesk(words1, 'bat', pos=wn.NOUN)
sense2 = lesk(words2, 'bat')

In [26]:
print("1. bat в первом предложении:", sense1.name())
print(" Определение:", sense1.definition())

print("\n2. bat во втором предложении:", sense2.name())
print(" Определение:", sense2.definition())

1. bat в первом предложении: bat.n.01
 Определение: nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate

2. bat во втором предложении: cricket_bat.n.01
 Определение: the club used in playing cricket


FastText + RuWordNet

In [6]:
# Импорт библиотек
from ruwordnet import RuWordNet
import numpy as np
import razdel

In [7]:
# Загружаем русскую версию WordNet
wn = RuWordNet()

In [8]:
# значения слова
for sense in wn.get_senses('замок'):
    print(sense.name, "-", sense.synset.definition)

ЗАМОК - здание (или комплекс зданий), обычно обнесённое стеной и сочетающее в себе оборонительную и жилую функции (в наиболее распространённом значении — укреплённое жилище феодала в средневековой Европе)
ЗАМОК - устройство для запирания чего-либо ключом


In [ ]:
from gensim.models import KeyedVectors
import numpy as np

# Загрузка модели
model_path = 'cc.ru.300.vec'
ft = KeyedVectors.load_word2vec_format(model_path, binary=False)

In [87]:
def vectorize(text):
    vecs = [ft[word] for word in text.lower().split() if word in ft and len(word) >= 3]
    if not vecs:
        return np.zeros(ft.vector_size)
    vec = np.mean(vecs, axis=0)
    return vec / np.linalg.norm(vec)


In [ ]:
# Выбираем наиболее подходящее значение многозначного слова по контексту
def disambiguate(word, context):
    candidates = [s.synset for s in wn[word]]
    context_vector = vectorize(' '.join(t.text for t in razdel.tokenize(context) if t.text.lower() != word.lower()))

    candidate_vectors = []
    for syn in candidates:
        synonyms = ' '.join(s.name for s in syn.senses)
        candidate_vectors.append(vectorize(synonyms))

    scores = [np.dot(context_vector, v) for v in candidate_vectors]
    best_synset = candidates[np.argmax(scores)]
    return best_synset.title, best_synset.definition


In [116]:
context1 = '''На вершине холма возвышался древний замок с башнями и бойницами.'''
disambiguate('замок', context1)


('СРЕДНЕВЕКОВЫЙ ЗАМОК',
 'здание (или комплекс зданий), обычно обнесённое стеной и сочетающее в себе оборонительную и жилую функции (в наиболее распространённом значении — укреплённое жилище феодала в средневековой Европе)')

In [117]:
context1 = '''Он щёлкнул ключом — и старый ржавый замок с лязгом открылся.'''
disambiguate('замок', context1)

('ЗАМОК ДЛЯ ЗАПИРАНИЯ', 'устройство для запирания чего-либо ключом')